In [1]:
import os
import sys
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("otto-baselines")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

from src.baselines import build_covisit_buys, popular_items, recommend
from src.metrics import recall_at_20

print("ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 14:36:51 WARN Utils: Your hostname, Omkars-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.75 instead (on interface en0)
26/11/25 14:36:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 14:36:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 14:36:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


ready: 4.3.0


In [2]:
from pyspark.sql import functions as F

tr = spark.read.parquet(str(DATA_PROCESSED / "split_train"))
ti = spark.read.parquet(str(DATA_PROCESSED / "test_input"))
tl = spark.read.parquet(str(DATA_PROCESSED / "test_labels"))

max_ts = spark.read.parquet(str(DATA_PROCESSED / "train_events")).agg(F.max("ts")).first()[0]
cutoff = max_ts - 7 * 24 * 3600 + 1

print("cutoff:", cutoff)
print("train events:", tr.count())
print("test sessions:", tl.count())

cutoff: 1661119200
train events: 163955181
test sessions: 1801251


In [3]:
popular = popular_items(tr, cutoff)
cv = build_covisit_buys(tr)
sessions = tl.select("session")

results = {
    "popular only": recall_at_20(recommend(ti, sessions, popular, use_history=False), tl),
    "history + popular": recall_at_20(recommend(ti, sessions, popular), tl),
    "history + covisit + popular": recall_at_20(recommend(ti, sessions, popular, covisit=cv), tl),
}

import pandas as pd
pd.DataFrame(results).T.round(4)

,clicks,carts,orders,score
popular only,0.0053,0.0067,0.0060,0.0061
history + popular,0.3259,0.3122,0.5736,0.4704
history + covisit + popular,0.4244,0.3831,0.6170,0.5276


In [6]:
seqs = (
    tr.groupBy("session")
    .agg(F.sort_array(F.collect_list(F.struct("ts", "aid"))).alias("s"))
    .select("session", F.transform("s", lambda x: x["aid"]).alias("all_items"))
    .withColumn(
        "dedup",
        F.expr("filter(all_items, (x, i) -> i = 0 OR x != all_items[i - 1])"),
    )
    .select(
        "session",
        F.expr("slice(dedup, greatest(1, size(dedup) - 19), 20)").alias("items"),
    )
    .filter(F.size("items") >= 2)
)

seqs.write.mode("overwrite").parquet(str(DATA_PROCESSED / "train_sequences"))

sq = spark.read.parquet(str(DATA_PROCESSED / "train_sequences"))
print("sequences:", sq.count())
sq.select(F.size("items").alias("len")).summary("mean", "50%", "max").show()
sq.show(3, truncate=100)

sequences: 9716693


+-------+-----------------+
|summary|              len|
+-------+-----------------+
|   mean|8.248530955953841|
|    50%|                5|
|    max|               20|
+-------+-----------------+

+-------+----------------------------------------------------------------------------------------------------+
|session|                                                                                               items|
+-------+----------------------------------------------------------------------------------------------------+
|     98|[593662, 932046, 1402537, 932046, 1673641, 932046, 24318, 932046, 1657142, 932046, 990324, 454744...|
|    195|                                                          [485296, 1187885, 706566, 1007308, 376552]|
|    256|[271070, 731352, 1357776, 718941, 731352, 1355234, 1417794, 1308169, 1461739, 1168475, 818093, 16...|
+-------+----------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [7]:
import numpy as np
import pyarrow.parquet as pq

L = 20

items = pq.read_table(DATA_PROCESSED / "train_sequences", columns=["items"]).column("items").combine_chunks()
offsets = items.offsets.to_numpy()
values = items.values.to_numpy().astype(np.int32) + 1  # shift IDs so 0 = padding
lengths = np.diff(offsets)

n = len(lengths)
X = np.zeros((n, L), dtype=np.int32)

rows = np.repeat(np.arange(n), lengths)
pos_in_seq = np.arange(len(values)) - np.repeat(offsets[:-1], lengths)
cols = pos_in_seq + (L - np.repeat(lengths, lengths))  # left padding
X[rows, cols] = values

np.save(DATA_PROCESSED / "train_seqs.npy", X)

print("shape:", X.shape, "| size:", round(X.nbytes / 1e9, 2), "GB")
print("largest ID:", X.max(), "| padding share:", round((X == 0).mean(), 3))
print(X[:3])

shape: (9716693, 20) | size: 0.78 GB
largest ID: 1855603 | padding share: 0.588
[[      0       0       0       0       0       0       0       0       0
        0       0       0       0       0  345694   33119  208147  360814
  1468794 1269913]
 [ 414414 1323310 1777316  734179  543396  368762  543396 1594166 1563669
   250385 1563669 1240414  654314 1240414  238805  908466 1240414  250385
  1230872 1509996]
 [1332590 1671548 1332590  339058  461212   29736 1436281  832193  373290
     9123 1706900 1227358  832193 1469979 1838330  758782  832193  137515
   832193 1436281]]


In [10]:
import pandas as pd
from pyspark.sql import Window

gru = spark.createDataFrame(
    pd.read_parquet(DATA_PROCESSED / "gru_candidates.parquet").assign(
        gru=lambda d: d["gru"].apply(lambda a: [int(x) for x in a])
    )
)

sessions = tl.select("session")
popular = popular_items(tr, cutoff)
cv = build_covisit_buys(tr)

hist = ti.groupBy("session", "aid").agg(F.max("ts").alias("last_ts"))
hist_items = (
    hist.withColumn("r", F.row_number().over(
        Window.partitionBy("session").orderBy(F.desc("last_ts"), "aid")))
    .filter("r <= 20")
    .groupBy("session")
    .agg(F.sort_array(F.collect_list(F.struct("r", "aid"))).alias("s"))
    .select("session", F.transform("s", lambda x: x["aid"]).alias("hist"))
)
covisit_preds = recommend(ti, sessions, popular, covisit=cv).withColumnRenamed("items", "cv_list")

empty = F.array().cast("array<int>")
pop = F.array(*[F.lit(a) for a in popular]).cast("array<int>")
base = sessions.join(hist_items, "session", "left").join(gru, "session", "left").join(covisit_preds, "session", "left")

def mix(*cols):
    return base.select(
        "session",
        F.slice(F.array_distinct(F.concat(*[F.coalesce(c, empty) for c in cols], pop)), 1, 20).alias("items"),
    )

results = {
    "history + covisit + popular": recall_at_20(mix("hist", "cv_list"), tl),
    "history + GRU + popular": recall_at_20(mix("hist", "gru"), tl),
    "history + GRU + covisit": recall_at_20(mix("hist", "gru", "cv_list"), tl),
    "history + covisit + GRU": recall_at_20(mix("hist", "cv_list", "gru"), tl),
}
pd.DataFrame(results).T.round(4)

26/11/25 14:16:15 WARN TaskSetManager: Stage 299 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.
26/11/25 14:17:30 WARN TaskSetManager: Stage 354 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.
26/11/25 14:18:34 WARN TaskSetManager: Stage 409 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.


[9968.403s][warning][gc,alloc] Executor task launch worker for task 9.0 in stage 420.0 (TID 2102): Retried waiting for GCLocker too often allocating 16777218 words


26/11/25 14:18:44 WARN TaskMemoryManager: Failed to allocate a page (134217728 bytes), try spilling task memory.
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.memory.HeapMemoryAllocator.allocate(HeapMemoryAllocator.java:72)
	at org.apache.spark.memory.TaskMemoryManager.allocatePageInternal(TaskMemoryManager.java:568)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:505)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:99)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:876)
	at org.apache.spark.unsafe.map.BytesToBytesMap.growAndRehash(BytesToBytesMap.java:999)
	at org.apache.spark.unsafe.map.BytesToBytesMap$Location.append(BytesToBytesMap.java:817)
	at org.apache.spark.sql.execution.UnsafeFixedWidthAggregationMap.getAggregationBufferFromUnsafeRow(UnsafeFixedWidthAggregationMap.java:135)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCode

[10251.815s][warning][gc,alloc] Executor task launch worker for task 8.0 in stage 700.0 (TID 4228): Retried waiting for GCLocker too often allocating 16777218 words


26/11/25 14:23:27 WARN TaskMemoryManager: Failed to allocate a page (134217728 bytes), try spilling task memory.
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.memory.HeapMemoryAllocator.allocate(HeapMemoryAllocator.java:72)
	at org.apache.spark.memory.TaskMemoryManager.allocatePageInternal(TaskMemoryManager.java:568)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:505)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:99)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:876)
	at org.apache.spark.unsafe.map.BytesToBytesMap.growAndRehash(BytesToBytesMap.java:999)
	at org.apache.spark.unsafe.map.BytesToBytesMap$Location.append(BytesToBytesMap.java:817)
	at org.apache.spark.sql.execution.UnsafeFixedWidthAggregationMap.getAggregationBufferFromUnsafeRow(UnsafeFixedWidthAggregationMap.java:135)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCode

,clicks,carts,orders,score
history + covisit + popular,0.4244,0.3831,0.6170,0.5276
history + GRU + popular,0.4507,0.3741,0.6081,0.5222
history + GRU + covisit,0.4507,0.3741,0.6081,0.5222
history + covisit + GRU,0.4244,0.3831,0.6170,0.5276


In [11]:
pool = base.select(
    "session",
    F.array_distinct(
        F.concat(F.coalesce("hist", empty), F.coalesce("gru", empty), F.coalesce("cv_list", empty))
    ).alias("items"),
)
print("average pool size:", pool.select(F.avg(F.size("items"))).first()[0])
print("pool recall:", recall_at_20(pool, tl))

26/11/25 14:27:47 WARN TaskSetManager: Stage 854 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.
26/11/25 14:29:01 WARN TaskSetManager: Stage 909 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.


average pool size: 54.61072221472743


26/11/25 14:30:10 WARN TaskSetManager: Stage 964 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.
26/11/25 14:31:21 WARN TaskSetManager: Stage 1019 contains a task of very large size (3212 KiB). The maximum recommended task size is 1000 KiB.


pool recall: {'clicks': 0.5314445148901531, 'carts': 0.43686478786839056, 'orders': 0.6457832251369935, 'score': 0.5716738229317285}


In [3]:
from src.split import make_test_labels, time_split

# weeks 1-2 vs week 3, using only data before the test week
tr2, val, cutoff2 = time_split(tr)
val_input, val_labels = make_test_labels(val, seed=7)

tr2.write.mode("overwrite").parquet(str(DATA_PROCESSED / "rank_train"))
val_input.write.mode("overwrite").parquet(str(DATA_PROCESSED / "rank_input"))
val_labels.write.mode("overwrite").parquet(str(DATA_PROCESSED / "rank_labels"))

r_tr = spark.read.parquet(str(DATA_PROCESSED / "rank_train"))
r_in = spark.read.parquet(str(DATA_PROCESSED / "rank_input"))
r_lb = spark.read.parquet(str(DATA_PROCESSED / "rank_labels"))

print("cutoff2:", cutoff2, "| one week before cutoff:", cutoff - 7 * 24 * 3600)
print("rank_train events:", r_tr.count())
print("rank sessions:", r_lb.count())
print("with carts:", r_lb.filter("carts is not null").count(),
      "| with orders:", r_lb.filter("orders is not null").count())

cutoff2: 1660514400 | one week before cutoff: 1660514400
rank_train events: 108548069
rank sessions: 2237925
with carts: 385933 | with orders: 196717


In [4]:
from pyspark.sql import Window

def candidates(inp, train_events, cut):
    """One row per (session, aid) candidate, with source scores."""
    w_recent = Window.partitionBy("session").orderBy(F.desc("last_ts"), "aid")
    hist = (
        inp.groupBy("session", "aid").agg(F.max("ts").alias("last_ts"))
        .withColumn("hist_rank", F.row_number().over(w_recent))
        .filter("hist_rank <= 20")
    )

    cv = build_covisit_buys(train_events)
    cov = (
        hist.join(cv, hist.aid == cv.aid_x)
        .groupBy("session", F.col("aid_y").alias("aid"))
        .agg(F.sum("count").alias("cov_score"))
        .withColumn("cov_rank", F.row_number().over(
            Window.partitionBy("session").orderBy(F.desc("cov_score"), "aid")))
        .filter("cov_rank <= 20")
    )

    top = popular_items(train_events, cut)
    pop = spark.createDataFrame([(a, r + 1) for r, a in enumerate(top)], "aid int, pop_rank int")
    sessions = inp.select("session").distinct()
    pop_c = sessions.crossJoin(pop)

    return (
        hist.select("session", "aid", "hist_rank")
        .join(cov.select("session", "aid", "cov_score", "cov_rank"), ["session", "aid"], "full")
        .join(pop_c, ["session", "aid"], "full")
    )

rank_cands = candidates(r_in, r_tr, cutoff2)
rank_cands.write.mode("overwrite").parquet(str(DATA_PROCESSED / "rank_candidates"))

rc = spark.read.parquet(str(DATA_PROCESSED / "rank_candidates"))
print("candidate rows:", rc.count())
print("per session:", rc.groupBy("session").count().agg(F.avg("count")).first()[0])
rc.orderBy("session", "hist_rank").show(10)

candidate rows: 88997745


per session: 39.767974798083046


+-------+-------+---------+---------+--------+--------+
|session|    aid|hist_rank|cov_score|cov_rank|pop_rank|
+-------+-------+---------+---------+--------+--------+
|8643220|1629608|     NULL|        5|      19|       5|
|8643220|1768884|     NULL|       22|       3|    NULL|
|8643220|1308823|     NULL|       47|       2|    NULL|
|8643220| 409620|     NULL|     NULL|    NULL|      18|
|8643220| 673407|     NULL|     NULL|    NULL|      10|
|8643220|1022566|     NULL|     NULL|    NULL|      12|
|8643220|  29735|     NULL|     NULL|    NULL|       4|
|8643220| 162064|     NULL|        4|      20|    NULL|
|8643220| 125278|     NULL|     NULL|    NULL|      16|
|8643220| 199994|     NULL|        7|       8|    NULL|
+-------+-------+---------+---------+--------+--------+
only showing top 10 rows


In [5]:
def add_features(cands, inp, train_events, cut):
    # this shopper's interactions with each item
    inter = inp.groupBy("session", "aid").agg(
        F.sum((F.col("type") == 0).cast("int")).alias("n_clicks"),
        F.sum((F.col("type") == 1).cast("int")).alias("n_carts"),
        F.sum((F.col("type") == 2).cast("int")).alias("n_orders"),
        F.max("ts").alias("item_last_ts"),
    )
    sess = inp.groupBy("session").agg(
        F.count("*").alias("sess_len"),
        F.countDistinct("aid").alias("sess_items"),
        F.max("ts").alias("sess_end"),
    )
    # item popularity in the week before the cutoff
    week = train_events.filter(F.col("ts") >= cut - 7 * 24 * 3600)
    item_pop = week.groupBy("aid").agg(
        F.sum((F.col("type") == 0).cast("int")).alias("item_clicks"),
        F.sum((F.col("type") == 1).cast("int")).alias("item_carts"),
        F.sum((F.col("type") == 2).cast("int")).alias("item_orders"),
    )
    return (
        cands.join(inter, ["session", "aid"], "left")
        .join(sess, "session", "left")
        .join(item_pop, "aid", "left")
        .withColumn("secs_since", F.col("sess_end") - F.col("item_last_ts"))
        .drop("item_last_ts", "sess_end")
        .fillna(0, subset=["n_clicks", "n_carts", "n_orders", "item_clicks", "item_carts", "item_orders"])
    )


def add_labels(df, labels):
    return (
        df.join(labels, "session", "left")
        .withColumn("y_click", (F.col("aid") == F.col("click")).cast("int"))
        .withColumn("y_cart", F.coalesce(F.array_contains("carts", F.col("aid")), F.lit(False)).cast("int"))
        .withColumn("y_order", F.coalesce(F.array_contains("orders", F.col("aid")), F.lit(False)).cast("int"))
        .fillna(0, subset=["y_click"])
        .drop("click", "carts", "orders")
    )


rank_data = add_labels(add_features(rc, r_in, r_tr, cutoff2), r_lb)
rank_data.write.mode("overwrite").parquet(str(DATA_PROCESSED / "rank_features"))

rf = spark.read.parquet(str(DATA_PROCESSED / "rank_features"))
print("rows:", rf.count())
rf.select(
    F.avg("y_click").alias("click_rate"),
    F.avg("y_cart").alias("cart_rate"),
    F.avg("y_order").alias("order_rate"),
    F.avg(F.col("hist_rank").isNotNull().cast("int")).alias("share_from_history"),
).show()
rf.filter("hist_rank is not null").orderBy("session", "hist_rank").show(5)

rows: 88997745
+-------------------+--------------------+--------------------+-------------------+
|         click_rate|           cart_rate|          order_rate| share_from_history|
+-------------------+--------------------+--------------------+-------------------+
|0.01314443416515778|0.004627544214743868|0.003002458095988...|0.07354949274276557|
+-------------------+--------------------+--------------------+-------------------+



+-------+-------+---------+---------+--------+--------+--------+-------+--------+--------+----------+-----------+----------+-----------+----------+-------+------+-------+
|session|    aid|hist_rank|cov_score|cov_rank|pop_rank|n_clicks|n_carts|n_orders|sess_len|sess_items|item_clicks|item_carts|item_orders|secs_since|y_click|y_cart|y_order|
+-------+-------+---------+---------+--------+--------+--------+-------+--------+--------+----------+-----------+----------+-----------+----------+-------+------+-------+
|8643220| 573273|        1|     NULL|    NULL|    NULL|       1|      0|       0|       1|         1|       4006|       340|        146|         0|      0|     0|      0|
|8643221| 921137|        1|     NULL|    NULL|    NULL|       1|      0|       0|       1|         1|         15|         1|          0|         0|      0|     0|      0|
|8643222|1037630|        1|     NULL|    NULL|    NULL|       1|      0|       0|       2|         2|        252|         1|          0|         